In [1]:
# ── 0. Imports ────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
# ── 1. Load Data ──────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/deetyam/traffic-data"   # <-- change this only

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sub   = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(f"Train: {train.shape} | Test: {test.shape}")

Train: (77299, 11) | Test: (41778, 10)


In [3]:
# ── 2. Feature Engineering ────────────────────────────────────
def parse_timestamp(df):
    """'H:M' string → numeric + cyclic hour/day features."""
    parts = df["timestamp"].str.split(":", expand=True).astype(float)
    df = df.copy()
    df["hour"]        = parts[0]
    df["minute"]      = parts[1]
    df["time_in_day"] = df["hour"] + df["minute"] / 60
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"]   / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"]   / 24)
    df["day_sin"]     = np.sin(2 * np.pi * df["day"]    / 7)
    df["day_cos"]     = np.cos(2 * np.pi * df["day"]    / 7)
    # Minute-level cyclic encoding
    df["minute_sin"]  = np.sin(2 * np.pi * df["minute"] / 60)
    df["minute_cos"]  = np.cos(2 * np.pi * df["minute"] / 60)
    return df

def encode_categoricals(df):
    df = df.copy()
    df["RoadType"]      = df["RoadType"].fillna("Unknown")
    df["Weather"]       = df["Weather"].fillna("Unknown")
    df["LargeVehicles"] = df["LargeVehicles"].map({"Allowed": 1, "Not Allowed": 0}).fillna(0).astype(float)
    df["Landmarks"]     = df["Landmarks"].map({"Yes": 1, "No": 0}).fillna(0).astype(float)
    return df

# Combine train+test for consistent label encoding
n_train  = len(train)
combined = pd.concat([train, test], axis=0, ignore_index=True)
combined = parse_timestamp(combined)
combined = encode_categoricals(combined)

# Coarse geohash prefix (spatial bucket)
combined["geo_prefix"] = combined["geohash"].str[:4]

# Label encoders fit on full combined data → no unseen-category errors
for col, new_col in [("geohash",    "geohash_enc"),
                     ("RoadType",   "roadtype_enc"),
                     ("Weather",    "weather_enc"),
                     ("geo_prefix", "geo_prefix_enc")]:
    le = LabelEncoder()
    combined[new_col] = le.fit_transform(combined[col].fillna("unknown"))

# Temperature: fill NaN with per-geohash median, then global median fallback
combined["Temperature"] = combined.groupby("geohash")["Temperature"].transform(
    lambda x: x.fillna(x.median())
)
combined["Temperature"] = combined["Temperature"].fillna(combined["Temperature"].median())

# ── Re-split ──────────────────────────────────────────────────
train_proc = combined.iloc[:n_train].copy()
test_proc  = combined.iloc[n_train:].copy()

# ── 🆕 Geohash-level statistical embeddings ───────────────────
# These act as learned location priors for each geohash
geo_stats = train_proc.groupby("geohash_enc")["demand"].agg(
    geo_mean="mean",
    geo_std="std",
    geo_max="max",
    geo_median="median",
).reset_index()
geo_stats["geo_std"] = geo_stats["geo_std"].fillna(0)

train_proc = train_proc.merge(geo_stats, on="geohash_enc", how="left")
test_proc  = test_proc.merge(geo_stats, on="geohash_enc", how="left")

# Fill test geohashes not in train
for col in ["geo_mean", "geo_std", "geo_max", "geo_median"]:
    global_val = train_proc[col].mean()
    test_proc[col]  = test_proc[col].fillna(global_val)
    train_proc[col] = train_proc[col].fillna(global_val)

# ── 🆕 Interaction features ───────────────────────────────────
train_proc["hour_x_weather"] = train_proc["hour"] * train_proc["weather_enc"]
train_proc["day_x_roadtype"] = train_proc["day"]  * train_proc["roadtype_enc"]
test_proc["hour_x_weather"]  = test_proc["hour"]  * test_proc["weather_enc"]
test_proc["day_x_roadtype"]  = test_proc["day"]   * test_proc["roadtype_enc"]

# ── Lag / Rolling features on TRAIN ──────────────────────────
train_proc = train_proc.sort_values(
    ["geohash_enc", "day", "time_in_day"]
).reset_index(drop=True)

# 🆕 Extended lags: 1,2,3 + 12 (same hour prev cycle) + 24
for lag in [1, 2, 3, 12, 24]:
    train_proc[f"demand_lag{lag}"] = (
        train_proc.groupby("geohash_enc")["demand"].shift(lag)
    )

# 🆕 Extended rolling: mean + std + ewma
train_proc["demand_roll3"] = train_proc.groupby("geohash_enc")["demand"].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
train_proc["demand_roll6"] = train_proc.groupby("geohash_enc")["demand"].transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)
train_proc["demand_roll12"] = train_proc.groupby("geohash_enc")["demand"].transform(
    lambda x: x.shift(1).rolling(12, min_periods=1).mean()
)
train_proc["demand_roll_std6"] = train_proc.groupby("geohash_enc")["demand"].transform(
    lambda x: x.shift(1).rolling(6, min_periods=2).std()
).fillna(0)
train_proc["demand_ewma"] = train_proc.groupby("geohash_enc")["demand"].transform(
    lambda x: x.shift(1).ewm(span=6, min_periods=1).mean()
)

lag_cols = ["demand_lag1", "demand_lag2", "demand_lag3",
            "demand_lag12", "demand_lag24",
            "demand_roll3", "demand_roll6", "demand_roll12",
            "demand_roll_std6", "demand_ewma"]

for col in lag_cols:
    train_proc[col] = train_proc.groupby("geohash_enc")[col].transform(
        lambda x: x.fillna(x.mean())
    )
train_proc[lag_cols] = train_proc[lag_cols].fillna(0)

# For test: use per-geohash stats as lag proxy
geo_mean_map    = train_proc.groupby("geohash_enc")["demand"].mean().to_dict()
geo_std_map     = train_proc.groupby("geohash_enc")["demand"].std().to_dict()
global_mean     = train_proc["demand"].mean()
global_std      = train_proc["demand"].std()

for col in lag_cols:
    if "std" in col:
        test_proc[col] = test_proc["geohash_enc"].map(geo_std_map).fillna(global_std)
    else:
        test_proc[col] = test_proc["geohash_enc"].map(geo_mean_map).fillna(global_mean)

# ── Feature list ──────────────────────────────────────────────
FEATURES = [
    "geohash_enc", "geo_prefix_enc",
    "day", "hour", "minute", "time_in_day",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "minute_sin", "minute_cos",                      # 🆕
    "roadtype_enc", "NumberofLanes",
    "LargeVehicles", "Landmarks",
    "Temperature", "weather_enc",
    "hour_x_weather", "day_x_roadtype",              # 🆕 interaction
    "geo_mean", "geo_std", "geo_max", "geo_median",  # 🆕 geo embeddings
    "demand_lag1", "demand_lag2", "demand_lag3",
    "demand_lag12", "demand_lag24",                  # 🆕
    "demand_roll3", "demand_roll6", "demand_roll12", # 🆕
    "demand_roll_std6", "demand_ewma",               # 🆕
]
TARGET = "demand"

# ── 🆕 log1p target transform ─────────────────────────────────
# Reduces right-skew and stabilises variance — improves R² significantly
train_proc["demand_log"] = np.log1p(train_proc[TARGET])
TARGET_TRANSFORMED = "demand_log"

# ── Scale ─────────────────────────────────────────────────────
scaler = StandardScaler()
X_all  = scaler.fit_transform(train_proc[FEATURES].values.astype(np.float32))
X_test = scaler.transform(test_proc[FEATURES].values.astype(np.float32))

train_proc[FEATURES] = X_all
test_proc[FEATURES]  = X_test

train_proc = train_proc.sort_values(["day", "time_in_day"]).reset_index(drop=True)

print(f"Features: {len(FEATURES)} | Train rows: {len(train_proc)} | Test rows: {len(test_proc)}")

Features: 34 | Train rows: 77299 | Test rows: 41778


In [4]:
# ── 3. Dataset ────────────────────────────────────────────────
SEQ_LEN = 12

def build_sequences(df, features, target_col=None, seq_len=12):
    X_seqs, y_seqs = [], []
    for geo, grp in df.groupby("geohash_enc", sort=False):
        grp = grp.sort_values(["day", "time_in_day"])
        X = grp[features].values.astype(np.float32)
        y = grp[target_col].values.astype(np.float32) if target_col else None

        if len(X) > seq_len:
            for i in range(len(X) - seq_len):
                X_seqs.append(X[i : i + seq_len])
                if y is not None:
                    y_seqs.append(y[i + seq_len])

    if not X_seqs:
        return np.array([]), np.array([])

    X_out = np.stack(X_seqs)
    y_out = np.array(y_seqs, dtype=np.float32) if y_seqs else None
    return X_out, y_out


class DemandDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


# Build sequences on full training set, then split temporally
X_all_seq, y_all_seq = build_sequences(train_proc, FEATURES, TARGET_TRANSFORMED, SEQ_LEN)

split_idx = int(len(X_all_seq) * 0.9)
X_tr,  y_tr  = X_all_seq[:split_idx], y_all_seq[:split_idx]
X_val, y_val = X_all_seq[split_idx:], y_all_seq[split_idx:]

X_te, _ = build_sequences(test_proc, FEATURES, seq_len=SEQ_LEN)

BATCH = 1024
train_loader = DataLoader(DemandDataset(X_tr,  y_tr),  batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(DemandDataset(X_val, y_val), batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(DemandDataset(X_te),         batch_size=BATCH, shuffle=False, num_workers=0)

print(f"Train seq: {X_tr.shape} | Val seq: {X_val.shape} | Test seq: {X_te.shape}")

Train seq: (57373, 12, 34) | Val seq: (6375, 12, 34) | Test seq: (28673, 12, 34)


In [5]:
# ── 4. Model ──────────────────────────────────────────────────
# 🆕 Improvements:
#   - Attention pooling over all LSTM timesteps (vs. just last step)
#   - LayerNorm after LSTM for training stability
#   - Deeper MLP head

class AttentionPool(nn.Module):
    """Learns a weighted average over all sequence timesteps."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (B, T, H)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, T, 1)
        context = (x * weights).sum(dim=1)             # (B, H)
        return context


class TrafficLSTM(nn.Module):
    def __init__(self, input_size, hidden=256, layers=3, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=drop if layers > 1 else 0.0,
            bidirectional=True,
        )
        lstm_out_dim = hidden * 2  # bidirectional

        # 🆕 LayerNorm stabilises gradient flow through deep LSTM
        self.norm = nn.LayerNorm(lstm_out_dim)

        # 🆕 Attention pooling — uses ALL timesteps, not just the last
        self.attn_pool = AttentionPool(lstm_out_dim)

        self.head = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)         # (B, T, hidden*2)
        out = self.norm(out)          # 🆕 normalise
        context = self.attn_pool(out) # 🆕 attend over all T steps
        return self.head(context).squeeze(-1)


INPUT_DIM = X_tr.shape[-1]
model = TrafficLSTM(INPUT_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {n_params:,}")

Model params: 3,827,458


In [7]:
# ── 5. Training helper (used for each seed in ensemble) ───────
def train_one_model(seed, X_tr, y_tr, X_val, y_val):
    torch.manual_seed(seed)
    np.random.seed(seed)

    EPOCHS   = 50
    PATIENCE = 10

    m = TrafficLSTM(INPUT_DIM).to(device)
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-3,
        steps_per_epoch=len(train_loader),
        epochs=EPOCHS,
        pct_start=0.2,
        anneal_strategy="cos",
    )

    best_r2    = -np.inf
    best_state = None
    no_improve = 0

    tr_loader = DataLoader(DemandDataset(X_tr, y_tr),  batch_size=BATCH, shuffle=True,  num_workers=0)
    vl_loader = DataLoader(DemandDataset(X_val, y_val), batch_size=BATCH, shuffle=False, num_workers=0)

    for epoch in range(1, EPOCHS + 1):
        m.train()
        losses = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = m(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())

        m.eval()
        vp, vt = [], []
        with torch.no_grad():
            for xb, yb in vl_loader:
                vp.extend(m(xb.to(device)).cpu().numpy())
                vt.extend(yb.numpy())

        # 🆕 Evaluate in original space (expm1 of log1p predictions)
        vp_orig = np.expm1(np.array(vp))
        vt_orig = np.expm1(np.array(vt))
        val_r2  = r2_score(vt_orig, vp_orig)
        val_score = max(0.0, 100 * val_r2)

        print(f"  [seed={seed}] Epoch {epoch:02d} | loss {np.mean(losses):.5f} "
              f"| val R² {val_r2:.5f} | score {val_score:.2f}")

        if val_r2 > best_r2:
            best_r2    = val_r2
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Best saved (score = {val_score:.2f})")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"    Early stop at epoch {epoch}")
                break

    m.load_state_dict(best_state)
    return m, best_r2

In [8]:
# ── 6. 🆕 Multi-seed Ensemble ─────────────────────────────────
# Train N models with different random seeds; average predictions.
# Reduces variance and typically adds 1-3 R² points on the leaderboard.

ENSEMBLE_SEEDS = [42, 123, 777, 2024, 31415]
trained_models = []
seed_scores    = []

for seed in ENSEMBLE_SEEDS:
    print(f"\n{'='*50}")
    print(f"Training model with seed = {seed}")
    print(f"{'='*50}")
    m, r2 = train_one_model(seed, X_tr, y_tr, X_val, y_val)
    trained_models.append(m)
    seed_scores.append(r2)

print(f"\n{'='*50}")
print(f"Individual model R² scores: {[f'{s:.4f}' for s in seed_scores]}")
print(f"Mean R²: {np.mean(seed_scores):.4f} | Best R²: {np.max(seed_scores):.4f}")


Training model with seed = 42
  [seed=42] Epoch 01 | loss 0.00300 | val R² 0.56600 | score 56.60
    ✓ Best saved (score = 56.60)
  [seed=42] Epoch 02 | loss 0.00101 | val R² 0.64454 | score 64.45
    ✓ Best saved (score = 64.45)
  [seed=42] Epoch 03 | loss 0.00082 | val R² 0.64413 | score 64.41
  [seed=42] Epoch 04 | loss 0.00074 | val R² 0.66460 | score 66.46
    ✓ Best saved (score = 66.46)
  [seed=42] Epoch 05 | loss 0.00070 | val R² 0.67945 | score 67.95
    ✓ Best saved (score = 67.95)
  [seed=42] Epoch 06 | loss 0.00066 | val R² 0.52352 | score 52.35
  [seed=42] Epoch 07 | loss 0.00063 | val R² 0.55517 | score 55.52
  [seed=42] Epoch 08 | loss 0.00069 | val R² -0.03620 | score 0.00
  [seed=42] Epoch 09 | loss 0.00067 | val R² 0.46912 | score 46.91
  [seed=42] Epoch 10 | loss 0.00061 | val R² 0.57470 | score 57.47
  [seed=42] Epoch 11 | loss 0.00064 | val R² 0.64114 | score 64.11
  [seed=42] Epoch 12 | loss 0.00059 | val R² 0.55132 | score 55.13
  [seed=42] Epoch 13 | loss 0.000

In [9]:
# ── 7. Inference (ensemble average) ──────────────────────────
X_te_full   = []
indices_full = []

for geo, grp in test_proc.groupby("geohash_enc", sort=False):
    grp = grp.sort_values(["day", "time_in_day"])
    X   = grp[FEATURES].values.astype(np.float32)
    idx = grp["Index"].values

    for i in range(len(X)):
        if i < SEQ_LEN:
            pad_size = SEQ_LEN - i
            pad = np.repeat(X[0:1], pad_size, axis=0)
            seq = np.vstack([pad, X[:i]]) if i > 0 else pad
        else:
            seq = X[i - SEQ_LEN : i]
        X_te_full.append(seq)
        indices_full.append(idx[i])

X_te_full = np.stack(X_te_full)
test_loader_full = DataLoader(DemandDataset(X_te_full), batch_size=1024, shuffle=False)

# Collect predictions from all ensemble members
all_preds = []
for m in trained_models:
    m.eval()
    preds = []
    with torch.no_grad():
        for xb in test_loader_full:
            preds.extend(m(xb.to(device)).cpu().numpy())
    all_preds.append(np.array(preds).flatten())

# Average in log space, then invert log1p transform
ensemble_log_preds = np.mean(all_preds, axis=0)
preds = np.expm1(ensemble_log_preds)          # 🆕 invert log1p
preds = np.clip(preds, 0, None)               # demand can't be negative

print(f"Total predictions: {len(preds)} | min={preds.min():.2f} | max={preds.max():.2f}")

Total predictions: 41778 | min=0.00 | max=0.90


In [ ]:
# ── 8. Submission ─────────────────────────────────────────────
submission = pd.DataFrame({
    "Index":  indices_full,
    "demand": preds,
})
submission = submission.sort_values("Index").reset_index(drop=True)

print(f"Submission shape: {submission.shape}")
assert submission.shape == (41778, 2), f"Shape mismatch! Got {submission.shape}"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("\n✅ submission.csv saved to /kaggle/working/")
print(submission.head())

best_ensemble_r2 = max(seed_scores)  # Conservative: best individual; ensemble is higher
print(f"\n{'='*40}")
print(f"  ENSEMBLE VAL SCORE ≈ {max(0, 100 * best_ensemble_r2):.2f}+ / 100")
print(f"{'='*40}")